In [1]:
# -------------------------------------------------------------
# Common pre‑amble – read config & export core variables
# -------------------------------------------------------------

from pathlib import Path
import os, sys

# infer repo root from the location of this file
repo_root = os.path.abspath('..')
sys.path.insert(0, str(repo_root))  # allow `import src.*`
from config.notebook_setup import *

Repository Root: /home/marcmaceira/projects/reuters-rag-classifier_clean_v2
Configuration: {'general': {'run_name': 'experiment_with_05_classes', 'seed': 42}, 'dataset': {'split_type': 'test', 'n_classes': 25, 'n_samples_per_class': 100}, 'paths': {'data_exploration_dir': 'output/experiment_with_05_classes/data_exploration', 'embeddings_dir': 'output/experiment_with_05_classes/embeddings', 'models_dir': 'output/experiment_with_05_classes/models', 'results_dir': 'output/experiment_with_05_classes/results'}, 'model': {'embedding_backend': 'sbert', 'sbert_model_name': 'sentence-transformers/all-MiniLM-L12-v2', 'openai_model_name': 'text-embedding-3-small', 'classifier': 'linear_svm', 'rag_top_k': 5, 'use_llm_refine': False}, 'training': {'batch_size': 32, 'max_epochs': 10, 'learning_rate': '1e-3'}, 'evaluation': {'metrics': ['accuracy', 'macro_f1', 'auc_ovr']}}

=== Configuration Variables ===

[DATASET]
  DATASET_N_CLASSES: 25
  DATASET_N_SAMPLES_PER_CLASS: 100
  DATASET_SPLIT_TYPE: test

# Reuters News Topic Classification - Exploratory Data Analysis

This notebook performs detailed exploratory data analysis on the Reuters news topic classification dataset. We'll analyze various aspects of the data to better understand its characteristics and potential challenges.

## Setup and Data Loading

In [2]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from src.datasets.dataset import get_dataset
from src.exploration import class_frequency, vocabulary_drift, length_distribution


In [3]:

# Load dataset
# X_train, y_train, _, _, _ = get_dataset(split_type="standard", n_classes=N_CLASSES)
X_train, y_train, X_test, y_test, classes = get_dataset(
    split_type=DATASET_SPLIT_TYPE,
    n_classes=N_CLASSES,
    n_samples_per_class=N_SAMPLES_PER_CLASS,
)

print(f"Loaded {len(X_train)} training documents with {N_CLASSES} classes")


INFO | Loading Reuters dataset with configuration:
INFO |   - Split type: test
INFO |   - Number of classes: 25
INFO |   - Samples per class: 100
INFO |   - Random seed: None
INFO | Loading small test dataset with 100 samples per class across 25 classes
INFO | Selected classes: earn, acq, crude, interest, money-fx, trade, grain, corn, dlr, money-supply, ship, coffee, sugar, gold, bop, gnp, cpi, cocoa, carcass, oilseed, copper, alum, reserves, jobs, barley
INFO |   - Class 'earn': 70 train, 30 test
INFO |   - Class 'acq': 70 train, 30 test
INFO |   - Class 'crude': 70 train, 30 test
INFO |   - Class 'interest': 70 train, 30 test
INFO |   - Class 'money-fx': 70 train, 30 test
INFO |   - Class 'trade': 70 train, 30 test
INFO |   - Class 'grain': 70 train, 30 test
INFO |   - Class 'corn': 70 train, 30 test
INFO |   - Class 'dlr': 70 train, 30 test
INFO |   - Class 'money-supply': 70 train, 30 test
INFO |   - Class 'ship': 70 train, 30 test
INFO |   - Class 'coffee': 70 train, 30 test
INFO 

Loaded 1511 training documents with 25 classes


## 1. Class Distribution Analysis

Let's analyze the distribution of topics in our dataset to understand class imbalance.

In [4]:

# Analyze class distribution
print("\n=== Class Distribution Analysis ===")
class_stats = class_frequency(
    labels=y_train,
    plot=True,
    save_path=os.path.join(DATA_EXPLORATION_DIR, "class_distribution.png"),
    top_n=N_CLASSES
)


=== Class Distribution Analysis ===


## 2. Document Length Analysis

Understanding the length distribution of articles helps us make decisions about preprocessing and model architecture.

In [5]:
# Plot length distribution
plt.figure(figsize=(10, 6))


stats = length_distribution(X_train, save_path=os.path.join(DATA_EXPLORATION_DIR, "document_length_distribution.png"), output_dir=DATA_EXPLORATION_DIR)
print(f"Mean length: {stats['stats']['mean']:.1f} tokens, median: {stats['stats']['median']}")

# Access percentile information from the returned stats
percentile_stats = stats['percentile_stats']


Document Length Percentiles:
25th percentile: 70 tokens
50th percentile: 114 tokens
75th percentile: 219 tokens
90th percentile: 382 tokens
95th percentile: 537 tokens
99th percentile: 759 tokens
Mean length: 173.7 tokens, median: 114.0


<Figure size 1000x600 with 0 Axes>

In [6]:
import os

# Plot length distribution
plt.figure(figsize=(10, 6))
stats = length_distribution(X_train, save_path=os.path.join(DATA_EXPLORATION_DIR, "document_length_distribution.png"))
print(f"Mean length: {stats['stats']['mean']:.1f} tokens, median: {stats['stats']['median']}")

# Calculate length percentiles
lengths = [len(doc.split()) for doc in X_train]
percentiles = np.percentile(lengths, [25, 50, 75, 90, 95, 99])
print("\nDocument Length Percentiles:")
length_stats = []
for p, percentile in zip([25, 50, 75, 90, 95, 99], percentiles):
    print(f"{p}th percentile: {percentile:.0f} tokens")
    length_stats.append({'percentile': p, 'length': int(percentile)})

# Save document length statistics to CSV
pd.DataFrame(length_stats).to_csv(os.path.join(DATA_EXPLORATION_DIR, "document_length_stats.csv"), index=False)

Mean length: 173.7 tokens, median: 114.0

Document Length Percentiles:
25th percentile: 70 tokens
50th percentile: 114 tokens
75th percentile: 219 tokens
90th percentile: 382 tokens
95th percentile: 538 tokens
99th percentile: 760 tokens


<Figure size 1000x600 with 0 Axes>

## 3. Vocabulary Analysis

Let's examine the vocabulary characteristics of our dataset.

In [7]:
#!/usr/bin/env python
"""
Simple script to run comprehensive vocabulary analysis on Reuters data.
"""
from src.exploration import comprehensive_analysis


# Run comprehensive analysis with custom settings
analysis_results = comprehensive_analysis(
    texts=X_train,                                          # Your text documents
    labels=y_train,                                         # Class labels for class-specific analysis
    label_names=classes,                                # Names of the classes 
    output_dir=os.path.join(DATA_EXPLORATION_DIR),                    # Output directory for results
    min_word_length=3,                                      # Minimum word length (filters short tokens)
    top_n=50,                                               # Number of top words to analyze
    create_visualizations=True,                             # Create and save plots
    create_csv=True                                         # Save results to CSV files
)

# The analysis_results dictionary contains all the results:
# - analysis_results['basic'] - basic analysis (only stopwords removed)
# - analysis_results['standard'] - standard filtering (stopwords, numbers, financial terms)
# - analysis_results['advanced'] - advanced filtering (adds domain-specific stopwords)
# - analysis_results['basic_class'] - class-specific words with basic filtering
# - analysis_results['standard_class'] - class-specific words with standard filtering
# - analysis_results['advanced_class'] - class-specific words with advanced filtering

print("\nAnalysis complete! Results saved to the 'vocab_analysis_custom' directory.")
print("You can also access the results programmatically through the returned dictionary.")

# Example: Get the top 5 words after advanced filtering
print("\nTop 5 words (advanced filtering):")
for word, count in analysis_results['advanced']['word_counts'].most_common(5):
    print(f"  {word}: {count:,}")

# Example: Get the most distinctive word for each class
print("\nMost distinctive word by class (advanced filtering):")
for class_name in classes:
    if class_name in analysis_results['advanced_class'].columns:
        top_word = analysis_results['advanced_class'][class_name][0]
        if top_word:  # Check that it's not an empty string
            print(f"  {class_name}: {top_word}") 

Running comprehensive vocabulary analysis on 1511 documents...

======= BASIC FILTERING =======
Total unique words (basic filtering): 12,645

======= STANDARD FILTERING =======
Total unique words (standard filtering): 11,380

======= ADVANCED FILTERING =======
Total unique words (advanced filtering): 11,380
Visualization saved to 'top_words_comprehensive.png'

======= CLASS-SPECIFIC ANALYSIS =======

Analysis complete! Results saved to the 'vocab_analysis_custom' directory.
You can also access the results programmatically through the returned dictionary.

Top 5 words (advanced filtering):
  year: 1,525
  would: 1,014
  trade: 916
  last: 785
  bank: 751

Most distinctive word by class (advanced filtering):
  earn: net (131)
  acq: company (75)
  crude: oil (239)
  interest: bank (145)
  money-fx: dollar (102)
  trade: trade (301)
  grain: wheat (188)
  corn: corn (199)
  dlr: dollar (262)
  money-supply: bank (99)
  ship: port (56)
  coffee: coffee (344)
  sugar: sugar (340)
  gold: go

## 4. Vocabulary drift



In [8]:

# Analyze vocabulary drift between train and test sets
print("\n=== Vocabulary Drift Analysis ===")
vocab_drift = vocabulary_drift(
    train_texts=X_train,
    test_texts=X_test,
    top_k=2000,
    min_freq=100,
    output_path=Path(RESULTS_DIR) / "vocabulary_drift.csv"
)

# Display top 10 tokens with highest drift
print("\nTop 20 tokens with highest frequency drift:")
display(vocab_drift.head(20))

# Display class distribution statistics
print("\nClass distribution statistics:")
display(class_stats['counts'])


=== Vocabulary Drift Analysis ===

Top 20 tokens with highest frequency drift:


,token,train_freq,test_freq,abs_diff
250,bundesbank,0.887324,0.112676,0.774648
48,debt,0.870000,0.130000,0.740000
224,gnp,0.850877,0.149123,0.701754
287,china,0.827815,0.172185,0.655629
65,marks,0.823529,0.176471,0.647059
420,meat,0.823529,0.176471,0.647059
0,german,0.816794,0.183206,0.633588
54,plan,0.816000,0.184000,0.632000
490,institute,0.811881,0.188119,0.623762
72,germany,0.811321,0.188679,0.622642



Class distribution statistics:


earn            70
acq             70
crude           70
interest        70
money-fx        70
trade           70
grain           70
corn            70
dlr             70
money-supply    70
ship            70
coffee          70
sugar           70
gold            70
bop             70
gnp             70
cpi             63
cocoa           50
carcass         46
oilseed         43
copper          42
alum            39
reserves        37
jobs            36
barley          35
Name: count, dtype: int64